In [1]:
import pandas as pd

In [2]:

# Cell 1: Pipeline Execution
print("Initiating Pipeline Execution...")

# 1. 模拟批量下载 (Binance Vision)
!python -m ingestion.bulk_download --pair BTCUSDT --interval 1h --start 2020-01

# 2. 模拟增量更新 (CCXT)
!python -m ingestion.incremental_update --pair BTCUSDT --interval 1h

# 3. 模拟数据调和与 Archive 生成
!python -m ingestion.reconcile --existing data/raw/btcusdt_1h.parquet --new data/raw/btcusdt_1h_update.parquet

print("Pipeline Execution Completed. Proceeding to Data QA...")

Initiating Pipeline Execution...
2026-04-16T01:12:15Z [INFO] Will download 76 months: 2020-01 to 2026-04
2026-04-16T01:12:15Z [INFO] Downloading https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-2020-01.zip ...
2026-04-16T01:12:16Z [INFO]   Parsed 744 rows for 2020-01
2026-04-16T01:12:16Z [INFO] Downloading https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-2020-02.zip ...
2026-04-16T01:12:18Z [INFO]   Parsed 690 rows for 2020-02
2026-04-16T01:12:18Z [INFO] Downloading https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-2020-03.zip ...
2026-04-16T01:12:19Z [INFO]   Parsed 743 rows for 2020-03
2026-04-16T01:12:19Z [INFO] Downloading https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-2020-04.zip ...
2026-04-16T01:12:21Z [INFO]   Parsed 718 rows for 2020-04
2026-04-16T01:12:21Z [INFO] Downloading https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-2020-05.zip ...
2026-04-

In [3]:
df = pd.read_parquet("data/raw/btcusdt_1h.parquet")
print(len(df))
print(df["open_time_utc"].min(), df["open_time_utc"].max())
print(df["source"].value_counts())

55105
2020-01-01 00:00:00+00:00 2026-04-16 07:00:00+00:00
source
binance_vision    54737
ccxt_binance        368
Name: count, dtype: int64[pyarrow]


In [4]:
# Cell 2: Setup and Schema Validation
import pandas as pd
import numpy as np
import glob
import json
import os
from pathlib import Path

canonical_path = Path("data/raw/btcusdt_1h.parquet")
quality_dir = Path("data/quality")
archive_dir = Path("data/raw/archive")

print("="*50)
print("🔍 CELL 2: CORE SCHEMA & DTYPES")
print("="*50)

if not canonical_path.exists():
    raise FileNotFoundError("❌ Canonical data not found! Bulk download failed.")

df = pd.read_parquet(canonical_path)

# 1. 检查时区
is_tz_aware = df['open_time_utc'].dt.tz is not None
is_utc = str(df['open_time_utc'].dt.tz) == 'UTC'
print(f"Timezone Aware? {'✅' if is_tz_aware else '❌'}")
print(f"Is strictly UTC? {'✅' if is_utc else '❌'}")

# 2. 检查对齐 (分钟和秒必须为0)
non_aligned = len(df[(df['open_time_utc'].dt.minute != 0) | (df['open_time_utc'].dt.second != 0)])
print(f"Perfectly hour-aligned? {'✅' if non_aligned == 0 else f'❌ ({non_aligned} unaligned rows)'}")

# 3. Source 分布
print("\n📊 Data Provenance Distribution:")
print(df['source'].value_counts(dropna=False).to_string())

🔍 CELL 2: CORE SCHEMA & DTYPES
Timezone Aware? ✅
Is strictly UTC? ✅
Perfectly hour-aligned? ✅

📊 Data Provenance Distribution:
source
binance_vision    54737
ccxt_binance        368


In [5]:
# Cell 3: Archive Integrity Check
print("="*50)
print("🗄️ CELL 3: ARCHIVE SNAPSHOT INTEGRITY")
print("="*50)

archives = sorted(glob.glob(str(archive_dir / "*.parquet")), key=os.path.getmtime, reverse=True)
has_ccxt = 'ccxt_api' in df['source'].values

if not has_ccxt and not archives:
    print("ℹ️ SKIPPED: 'ccxt_api' source not found. Reconcile hasn't processed new data. No archive expected.")
elif has_ccxt and not archives:
    print("❌ FAIL: 'ccxt_api' data exists but NO ARCHIVE found! Reconcile overwrote without backup!")
elif archives:
    latest_archive = archives[0]
    archive_name = Path(latest_archive).name
    print(f"✅ PASS: Archive successfully created -> {archive_name}")
    
    # 检查 ISO 时间戳命名规范
    has_timestamp_format = 'T' in archive_name and 'Z' in archive_name
    print(f"   -> ISO Timestamp format (T/Z)? {'✅' if has_timestamp_format else '❌ (Check reconcile.py)'}")
    
    # 对比行数
    df_archive = pd.read_parquet(latest_archive)
    diff = len(df) - len(df_archive)
    print(f"\n   -> Canonical Rows: {len(df)}")
    print(f"   -> Archive Rows:   {len(df_archive)}")
    if diff > 0:
        print(f"✅ PASS: Canonical has {diff} new rows compared to Archive.")
    elif diff == 0:
        print("⚠️ WARNING: Rows unchanged. CCXT API might not have fetched new data.")
    else:
        print("❌ FAIL: Canonical is SMALLER than Archive. Data was dropped during reconcile!")

🗄️ CELL 3: ARCHIVE SNAPSHOT INTEGRITY
✅ PASS: Archive successfully created -> btcusdt_1h_20260416T081415Z.parquet
   -> ISO Timestamp format (T/Z)? ✅

   -> Canonical Rows: 55105
   -> Archive Rows:   54737
✅ PASS: Canonical has 368 new rows compared to Archive.


In [6]:
# Cell 4: Validation Report Audit
print("="*50)
print("📑 CELL 4: VALIDATOR JSON AUDIT")
print("="*50)

reports = sorted(glob.glob(str(quality_dir / "*validation*.json")), key=os.path.getmtime, reverse=True)

if not reports:
    print("❌ FAIL: No validation report JSON found.")
else:
    with open(reports[0], 'r') as f:
        report_data = json.load(f)
        print(f"Latest Report: {Path(reports[0]).name}")
        print(f"Overall Status: {report_data.get('overall_status', 'UNKNOWN')}\n")
        
        checks = report_data.get('checks', {})
        
        # 提取关键警告
        zero_vol = checks.get('zero_volume_bars', {}).get('count', 0)
        vol_anomaly = checks.get('volume_anomaly', {}).get('count', 0)
        price_sanity = checks.get('price_sanity', {}).get('count', 0)
        
        print(f"⚠️ Zero-Volume Bars: {zero_vol}")
        print(f"⚠️ Volume Drops (>95%): {vol_anomaly}")
        print(f"⚠️ Extreme Price Moves: {price_sanity}")
        
        if zero_vol > 0:
            print("   -> Tip: These are likely exchange downtimes. Keep 'flag_only' per execution.yaml.")

📑 CELL 4: VALIDATOR JSON AUDIT
Latest Report: reconcile_validation_20260416.json
Overall Status: WARNING

⚠️ Zero-Volume Bars: 3
⚠️ Volume Drops (>95%): 0
⚠️ Extreme Price Moves: 0
   -> Tip: These are likely exchange downtimes. Keep 'flag_only' per execution.yaml.


In [ ]:
# Cell 5: Monthly Data Gap Deep Dive

# Load canonical dataset
df = pd.read_parquet("data/raw/btcusdt_1h.parquet").copy()
df = df.sort_values("open_time_utc").reset_index(drop=True)

# Sanity
assert df["open_time_utc"].is_monotonic_increasing, "Timestamps are not sorted"
assert str(df["open_time_utc"].dt.tz) == "UTC", "Timestamps are not UTC"

start_time = df["open_time_utc"].min()
end_time = df["open_time_utc"].max()

# Build perfect expected hourly index
expected_index = pd.date_range(
    start=start_time,
    end=end_time,
    freq="1h",
    tz="UTC",
)

# Find exact missing timestamps
actual_index = pd.DatetimeIndex(df["open_time_utc"])
missing_index = expected_index.difference(actual_index)

print("=" * 60)
print("MISSING ROW SUMMARY")
print("=" * 60)
print(f"Start:         {start_time}")
print(f"End:           {end_time}")
print(f"Expected rows: {len(expected_index)}")
print(f"Actual rows:   {len(df)}")
print(f"Missing rows:  {len(missing_index)}")

if len(missing_index) == 0:
    print("\n✅ No missing rows.")
else:
    # Convert to Series for grouping consecutive hours
    missing_series = pd.Series(missing_index, name="missing_time_utc")

    # New group whenever the gap between consecutive missing timestamps is > 1 hour
    group_breaks = missing_series.diff() > pd.Timedelta(hours=1)
    group_id = group_breaks.cumsum()

    intervals = (
        missing_series.groupby(group_id)
        .agg(["min", "max", "count"])
        .rename(columns={"min": "gap_start_utc", "max": "gap_end_utc", "count": "missing_hours"})
        .reset_index(drop=True)
    )

    print("\nMISSING INTERVALS")
    print(intervals.to_string(index=False))

    # Optional: show every missing timestamp
    missing_df = pd.DataFrame({"missing_time_utc": missing_index})
    print("\nFIRST 20 MISSING TIMESTAMPS")
    print(missing_df.head(20).to_string(index=False))

MISSING ROW SUMMARY
Start:         2020-01-01 00:00:00+00:00
End:           2026-04-16 07:00:00+00:00
Expected rows: 55136
Actual rows:   55105
Missing rows:  31

MISSING INTERVALS
            gap_start_utc               gap_end_utc  missing_hours
2020-02-09 02:00:00+00:00 2020-02-09 02:00:00+00:00              1
2020-02-19 12:00:00+00:00 2020-02-19 16:00:00+00:00              5
2020-03-04 10:00:00+00:00 2020-03-04 10:00:00+00:00              1
2020-04-25 02:00:00+00:00 2020-04-25 03:00:00+00:00              2
2020-06-28 02:00:00+00:00 2020-06-28 04:00:00+00:00              3
2020-11-30 06:00:00+00:00 2020-11-30 06:00:00+00:00              1
2020-12-21 15:00:00+00:00 2020-12-21 17:00:00+00:00              3
2020-12-25 02:00:00+00:00 2020-12-25 02:00:00+00:00              1
2021-02-11 04:00:00+00:00 2021-02-11 04:00:00+00:00              1
2021-03-06 02:00:00+00:00 2021-03-06 02:00:00+00:00              1
2021-04-20 02:00:00+00:00 2021-04-20 03:00:00+00:00              2
2021-04-25 05:0

In [2]:
# --- Nearby context around each missing interval ---
print("\n" + "=" * 60)
print("NEARBY CONTEXT AROUND EACH GAP")
print("=" * 60)

context_hours = 3  # rows before/after each gap window

for i, row in intervals.iterrows():
    gap_start = row["gap_start_utc"]
    gap_end = row["gap_end_utc"]

    window_start = gap_start - pd.Timedelta(hours=context_hours)
    window_end = gap_end + pd.Timedelta(hours=context_hours)

    context = df[
        (df["open_time_utc"] >= window_start) &
        (df["open_time_utc"] <= window_end)
    ][["open_time_utc", "open", "high", "low", "close", "volume", "source"]]

    print(f"\n--- GAP {i+1} ---")
    print(f"Gap: {gap_start} to {gap_end} ({row['missing_hours']} missing hours)")
    print(context.to_string(index=False))


NEARBY CONTEXT AROUND EACH GAP

--- GAP 1 ---
Gap: 2020-02-09 02:00:00+00:00 to 2020-02-09 02:00:00+00:00 (1 missing hours)
            open_time_utc     open     high      low    close      volume         source
2020-02-08 23:00:00+00:00  9913.26  9923.00  9892.50  9895.05  832.750529 binance_vision
2020-02-09 00:00:00+00:00  9895.04  9947.50  9880.75  9936.95 1430.944116 binance_vision
2020-02-09 01:00:00+00:00  9937.05  9944.25  9909.56  9938.90  957.502674 binance_vision
2020-02-09 03:00:00+00:00  9938.86 10052.67  9937.92 10052.63 4894.060253 binance_vision
2020-02-09 04:00:00+00:00 10052.66 10131.00 10052.22 10091.27 3942.411537 binance_vision
2020-02-09 05:00:00+00:00 10091.85 10114.25 10040.00 10099.99 2139.404330 binance_vision

--- GAP 2 ---
Gap: 2020-02-19 12:00:00+00:00 to 2020-02-19 16:00:00+00:00 (5 missing hours)
            open_time_utc     open     high      low    close      volume         source
2020-02-19 09:00:00+00:00 10158.20 10159.53 10101.01 10112.99 1655.486

In [3]:
# --- Zero-volume bars ---
zero_vol = df[df["volume"] == 0][["open_time_utc", "open", "high", "low", "close", "volume", "source"]]

print("\n" + "=" * 60)
print("ZERO-VOLUME BARS")
print("=" * 60)
if zero_vol.empty:
    print("No zero-volume bars.")
else:
    print(zero_vol.to_string(index=False))

# Mark if each zero-volume bar is within N hours of a missing interval
if len(missing_index) > 0 and not zero_vol.empty:
    proximity_hours = 6
    gap_hours_set = set(missing_index)

    def near_gap(ts):
        for h in range(-proximity_hours, proximity_hours + 1):
            if ts + pd.Timedelta(hours=h) in gap_hours_set:
                return True
        return False

    zero_vol = zero_vol.copy()
    zero_vol["within_6h_of_missing_gap"] = zero_vol["open_time_utc"].apply(near_gap)

    print("\nZERO-VOLUME BARS WITH GAP PROXIMITY")
    print(zero_vol.to_string(index=False))


ZERO-VOLUME BARS
            open_time_utc     open     high      low    close  volume         source
2020-12-21 14:00:00+00:00 22646.53 22646.53 22646.53 22646.53     0.0 binance_vision
2021-02-11 03:00:00+00:00 44582.07 44582.07 44582.07 44582.07     0.0 binance_vision
2023-03-24 12:00:00+00:00 28080.00 28080.00 28080.00 28080.00     0.0 binance_vision

ZERO-VOLUME BARS WITH GAP PROXIMITY
            open_time_utc     open     high      low    close  volume         source  within_6h_of_missing_gap
2020-12-21 14:00:00+00:00 22646.53 22646.53 22646.53 22646.53     0.0 binance_vision                      True
2021-02-11 03:00:00+00:00 44582.07 44582.07 44582.07 44582.07     0.0 binance_vision                      True
2023-03-24 12:00:00+00:00 28080.00 28080.00 28080.00 28080.00     0.0 binance_vision                      True


In [4]:
# --- Missing rows by month ---
if len(missing_index) > 0:
    missing_by_month = (
        pd.Series(1, index=missing_index)
        .resample("ME")
        .sum()
        .fillna(0)
        .astype(int)
        .rename("missing_rows")
    )

    missing_by_month = missing_by_month[missing_by_month > 0]

    print("\n" + "=" * 60)
    print("MISSING ROWS BY MONTH")
    print("=" * 60)
    print(missing_by_month.to_string())


MISSING ROWS BY MONTH
2020-02-29 00:00:00+00:00    6
2020-03-31 00:00:00+00:00    1
2020-04-30 00:00:00+00:00    2
2020-06-30 00:00:00+00:00    3
2020-11-30 00:00:00+00:00    1
2020-12-31 00:00:00+00:00    4
2021-02-28 00:00:00+00:00    1
2021-03-31 00:00:00+00:00    1
2021-04-30 00:00:00+00:00    5
2021-08-31 00:00:00+00:00    4
2021-09-30 00:00:00+00:00    2
2023-03-31 00:00:00+00:00    1


In [5]:
# --- Source coverage summary ---
print("\n" + "=" * 60)
print("SOURCE COVERAGE")
print("=" * 60)
print(df["source"].value_counts(dropna=False).to_string())

for src in df["source"].dropna().unique():
    sub = df[df["source"] == src]
    print(f"\nSource: {src}")
    print(f"  Rows:  {len(sub)}")
    print(f"  Start: {sub['open_time_utc'].min()}")
    print(f"  End:   {sub['open_time_utc'].max()}")


SOURCE COVERAGE
source
binance_vision    54737
ccxt_binance        368

Source: binance_vision
  Rows:  54737
  Start: 2020-01-01 00:00:00+00:00
  End:   2026-03-31 23:00:00+00:00

Source: ccxt_binance
  Rows:  368
  Start: 2026-04-01 00:00:00+00:00
  End:   2026-04-16 07:00:00+00:00


In [6]:
summary = {
    "start_utc": df["open_time_utc"].min(),
    "end_utc": df["open_time_utc"].max(),
    "expected_rows": len(expected_index),
    "actual_rows": len(df),
    "missing_rows": len(missing_index),
    "duplicate_timestamps": int(df["open_time_utc"].duplicated().sum()),
    "zero_volume_rows": int((df["volume"] == 0).sum()),
    "sources": df["source"].value_counts(dropna=False).to_dict(),
}
print(summary)

{'start_utc': Timestamp('2020-01-01 00:00:00+0000', tz='UTC'), 'end_utc': Timestamp('2026-04-16 07:00:00+0000', tz='UTC'), 'expected_rows': 55136, 'actual_rows': 55105, 'missing_rows': 31, 'duplicate_timestamps': 0, 'zero_volume_rows': 3, 'sources': {'binance_vision': 54737, 'ccxt_binance': 368}}


In [1]:
# =============================================================
# BTC ALPHA PIPELINE — PHASE 0 SPOT CHECK
# =============================================================
# Run this in a Jupyter notebook after loading the canonical dataset.
# Purpose: Human verification that prices match reality at known events.
# This is the final Phase 0 data sign-off step.
# =============================================================

import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_parquet("data/raw/btcusdt_1h.parquet")


In [2]:

# =============================================================
# SPOT CHECK 1: Known Market Events
# =============================================================
# We check that prices at known historical moments are within
# a reasonable range of what actually happened.
# Sources: CoinGecko, TradingView, public record.

print("=" * 70)
print("SPOT CHECK 1: KNOWN MARKET EVENTS")
print("=" * 70)

known_events = [
    {
        "name": "COVID Crash — Black Thursday",
        "date": "2020-03-12",
        "check": "daily_low",
        "expected_range": (3800, 5500),
        "notes": "BTC crashed from ~$7,900 to ~$3,800 intraday"
    },
    {
        "name": "COVID Crash — Recovery Start",
        "date": "2020-03-13",
        "check": "daily_close_approx",
        "expected_range": (5000, 6000),
        "notes": "Bounce day after the crash"
    },
    {
        "name": "2020 Halving Day",
        "date": "2020-05-11",
        "check": "daily_close_approx",
        "expected_range": (8500, 9200),
        "notes": "Third BTC halving, price ~$8,600-$8,800"
    },
    {
        "name": "BTC First Hits $20K (2020 Bull Run)",
        "date": "2020-12-16",
        "check": "daily_high",
        "expected_range": (20000, 21500),
        "notes": "First time surpassing 2017 ATH"
    },
    {
        "name": "BTC ATH Window — Nov 2021",
        "date": "2021-11-10",
        "check": "daily_high",
        "expected_range": (66000, 69500),
        "notes": "All-time high ~$69,000 on Nov 10, 2021"
    },
    {
        "name": "LUNA/UST Collapse",
        "date": "2022-05-12",
        "check": "daily_low",
        "expected_range": (26000, 30000),
        "notes": "Terra collapse caused BTC to crash to ~$26K-$27K"
    },
    {
        "name": "FTX Collapse — Nov 2022",
        "date": "2022-11-09",
        "check": "daily_low",
        "expected_range": (15500, 18000),
        "notes": "FTX bankruptcy, BTC dropped to ~$15.5K-$17K"
    },
    {
        "name": "Cycle Low — Post-FTX",
        "date": "2022-11-21",
        "check": "daily_low",
        "expected_range": (15400, 16500),
        "notes": "Approximate cycle bottom around $15,400-$15,800"
    },
    {
        "name": "BTC ETF Approval Day",
        "date": "2024-01-11",
        "check": "daily_high",
        "expected_range": (46000, 49500),
        "notes": "SEC approved spot BTC ETFs, price ~$46K-$49K"
    },
    {
        "name": "2024 Halving Day",
        "date": "2024-04-20",
        "check": "daily_close_approx",
        "expected_range": (63000, 66000),
        "notes": "Fourth BTC halving, price ~$64K"
    },
]

results = []
for event in known_events:
    date = pd.Timestamp(event["date"], tz="UTC")
    day_data = df[df["open_time_utc"].dt.date == date.date()]

    if day_data.empty:
        status = "❌ NO DATA"
        actual = None
    else:
        if event["check"] == "daily_low":
            actual = day_data["low"].min()
        elif event["check"] == "daily_high":
            actual = day_data["high"].max()
        elif event["check"] == "daily_close_approx":
            actual = day_data.iloc[-1]["close"]
        else:
            actual = day_data.iloc[-1]["close"]

        lo, hi = event["expected_range"]
        if lo <= actual <= hi:
            status = "✅ PASS"
        else:
            status = f"⚠️ OUT OF RANGE"

    results.append({
        "Event": event["name"],
        "Date": event["date"],
        "Check": event["check"],
        "Expected": f"${event['expected_range'][0]:,.0f} - ${event['expected_range'][1]:,.0f}",
        "Actual": f"${actual:,.2f}" if actual else "N/A",
        "Status": status,
    })

results_df = pd.DataFrame(results)
for _, row in results_df.iterrows():
    print(f"\n{row['Status']}  {row['Event']}")
    print(f"   Date: {row['Date']}  |  Check: {row['Check']}")
    print(f"   Expected: {row['Expected']}  |  Actual: {row['Actual']}")

passed = sum(1 for r in results if "PASS" in r["Status"])
total = len(results)
print(f"\n{'=' * 70}")
print(f"RESULT: {passed}/{total} events verified")
print(f"{'=' * 70}")



SPOT CHECK 1: KNOWN MARKET EVENTS

✅ PASS  COVID Crash — Black Thursday
   Date: 2020-03-12  |  Check: daily_low
   Expected: $3,800 - $5,500  |  Actual: $4,410.00

✅ PASS  COVID Crash — Recovery Start
   Date: 2020-03-13  |  Check: daily_close_approx
   Expected: $5,000 - $6,000  |  Actual: $5,578.60

✅ PASS  2020 Halving Day
   Date: 2020-05-11  |  Check: daily_close_approx
   Expected: $8,500 - $9,200  |  Actual: $8,561.52

⚠️ OUT OF RANGE  BTC First Hits $20K (2020 Bull Run)
   Date: 2020-12-16  |  Check: daily_high
   Expected: $20,000 - $21,500  |  Actual: $21,560.00

✅ PASS  BTC ATH Window — Nov 2021
   Date: 2021-11-10  |  Check: daily_high
   Expected: $66,000 - $69,500  |  Actual: $69,000.00

✅ PASS  LUNA/UST Collapse
   Date: 2022-05-12  |  Check: daily_low
   Expected: $26,000 - $30,000  |  Actual: $26,700.00

✅ PASS  FTX Collapse — Nov 2022
   Date: 2022-11-09  |  Check: daily_low
   Expected: $15,500 - $18,000  |  Actual: $15,588.00

✅ PASS  Cycle Low — Post-FTX
   Date: 

In [3]:

# =============================================================
# SPOT CHECK 2: Basic Statistical Sanity
# =============================================================
print("\n\n" + "=" * 70)
print("SPOT CHECK 2: STATISTICAL SANITY")
print("=" * 70)

# Yearly summary — does the price trajectory make sense?
df_with_year = df.copy()
df_with_year["year"] = df_with_year["open_time_utc"].dt.year

yearly = df_with_year.groupby("year").agg(
    rows=("close", "size"),
    open_price=("open", "first"),
    close_price=("close", "last"),
    year_high=("high", "max"),
    year_low=("low", "min"),
    avg_hourly_volume=("volume", "mean"),
).round(2)

yearly["year_return_pct"] = ((yearly["close_price"] / yearly["open_price"] - 1) * 100).round(1)

print("\nYearly Summary:")
print(yearly.to_string())

# Known approximate year-end closes for sanity:
# 2020: ~$29K, 2021: ~$46K, 2022: ~$16.5K, 2023: ~$42K, 2024: ~$93K
print("\nExpected approximate year-end closes:")
print("  2020: ~$29,000  |  2021: ~$46,000  |  2022: ~$16,500")
print("  2023: ~$42,000  |  2024: ~$93,000  |  2025: check current")





SPOT CHECK 2: STATISTICAL SANITY

Yearly Summary:
      rows  open_price  close_price  year_high  year_low  avg_hourly_volume  year_return_pct
year                                                                                        
2020  8767     7195.24     28923.63   29300.00   3782.13            2935.19            302.0
2021  8747    28923.63     46216.93   69000.00  28130.00            2917.58             59.8
2022  8760    46216.93     16542.40   48189.84  15476.00            6101.73            -64.2
2023  8759    16541.77     42283.58   44700.00  16499.01            4179.17            155.6
2024  8784    42283.58     93576.00  108353.00  38555.00            1472.12            121.3
2025  8760    93576.00     87648.22  126199.63  74508.00             880.20             -6.3
2026  2528    87648.21     74707.03   97924.49  60000.00             908.33            -14.8

Expected approximate year-end closes:
  2020: ~$29,000  |  2021: ~$46,000  |  2022: ~$16,500
  2023: ~$42,000 

In [5]:

# =============================================================
# SPOT CHECK 3: Source Transition Boundary
# =============================================================
print("\n\n" + "=" * 70)
print("SPOT CHECK 3: SOURCE TRANSITION BOUNDARY")
print("=" * 70)
print("Verifying clean handoff between binance_vision and ccxt_binance\n")

# Find the transition point
bv_last = df[df["source"] == "binance_vision"]["open_time_utc"].max()
ccxt_first = df[df["source"] == "ccxt_binance"]["open_time_utc"].min()

print(f"Last binance_vision row:  {bv_last}")
print(f"First ccxt_binance row:   {ccxt_first}")

gap_hours = (ccxt_first - bv_last).total_seconds() / 3600
print(f"Gap between sources:      {gap_hours:.0f} hours", end="")
if gap_hours == 1:
    print("  ✅ Perfect handoff (exactly 1 hour apart)")
elif gap_hours == 0:
    print("  ⚠️ Overlap — check for duplicate handling")
else:
    print(f"  ⚠️ {gap_hours:.0f}h gap — investigate")

# Show the boundary rows
boundary = df[
    (df["open_time_utc"] >= bv_last - pd.Timedelta(hours=2)) &
    (df["open_time_utc"] <= ccxt_first + pd.Timedelta(hours=2))
][["open_time_utc", "open", "high", "low", "close", "volume", "source"]]
print(f"\nBoundary rows:")
print(boundary.to_string(index=False))

# Price continuity at boundary
bv_last_close = df[df["open_time_utc"] == bv_last]["close"].values[0]
ccxt_first_open = df[df["open_time_utc"] == ccxt_first]["open"].values[0]
pct_diff = abs(ccxt_first_open / bv_last_close - 1) * 100
print(f"\nPrice continuity: binance_vision close=${bv_last_close:,.2f} → ccxt_binance open=${ccxt_first_open:,.2f}")
print(f"Difference: {pct_diff:.3f}%", end="")
if pct_diff < 1.0:
    print("  ✅ Normal")
else:
    print("  ⚠️ Large gap — may indicate different exchange or data issue")





SPOT CHECK 3: SOURCE TRANSITION BOUNDARY
Verifying clean handoff between binance_vision and ccxt_binance

Last binance_vision row:  2026-03-31 23:00:00+00:00
First ccxt_binance row:   2026-04-01 00:00:00+00:00
Gap between sources:      1 hours  ✅ Perfect handoff (exactly 1 hour apart)

Boundary rows:
            open_time_utc     open     high      low    close     volume         source
2026-03-31 21:00:00+00:00 68265.53 68410.30 67884.00 68277.99  494.48096 binance_vision
2026-03-31 22:00:00+00:00 68278.00 68341.16 67962.27 68172.87  755.55219 binance_vision
2026-03-31 23:00:00+00:00 68173.59 68371.10 68090.20 68284.48  610.99275 binance_vision
2026-04-01 00:00:00+00:00 68284.49 68343.77 67853.30 68330.75  542.79511   ccxt_binance
2026-04-01 01:00:00+00:00 68330.76 68378.64 67760.33 67782.73  966.68117   ccxt_binance
2026-04-01 02:00:00+00:00 67782.74 67913.98 67578.75 67668.89 1114.17912   ccxt_binance

Price continuity: binance_vision close=$68,284.48 → ccxt_binance open=$68,284.4

In [6]:

# =============================================================
# SPOT CHECK 4: Gap Documentation (for known_anomalies record)
# =============================================================
print("\n\n" + "=" * 70)
print("SPOT CHECK 4: GAP DOCUMENTATION FOR KNOWN ANOMALIES")
print("=" * 70)

expected_index = pd.date_range(
    start=df["open_time_utc"].min(),
    end=df["open_time_utc"].max(),
    freq="1h",
    tz="UTC",
)
missing_index = expected_index.difference(df["open_time_utc"])

if len(missing_index) == 0:
    print("No gaps found.")
else:
    # Group consecutive missing hours into intervals
    missing_sorted = sorted(missing_index)
    intervals = []
    current_start = missing_sorted[0]
    current_end = missing_sorted[0]

    for ts in missing_sorted[1:]:
        if ts == current_end + pd.Timedelta(hours=1):
            current_end = ts
        else:
            intervals.append((current_start, current_end))
            current_start = ts
            current_end = ts
    intervals.append((current_start, current_end))

    print(f"\n{len(missing_index)} missing hours in {len(intervals)} gap windows:\n")
    print(f"{'#':<4} {'Start (UTC)':<28} {'End (UTC)':<28} {'Hours':<6} {'Likely Cause'}")
    print("-" * 90)

    for i, (start, end) in enumerate(intervals):
        hours = int((end - start).total_seconds() / 3600) + 1
        # Check if adjacent to zero-volume bar
        nearby_zero = df[
            (df["volume"] == 0) &
            (df["open_time_utc"] >= start - pd.Timedelta(hours=6)) &
            (df["open_time_utc"] <= end + pd.Timedelta(hours=6))
        ]
        cause = "Exchange downtime" if not nearby_zero.empty else "Missing data (unknown)"
        print(f"{i+1:<4} {str(start):<28} {str(end):<28} {hours:<6} {cause}")

    print(f"\nAll gaps in 2020-2023 range: {'✅ YES' if all(s.year <= 2023 for s, _ in intervals) else '❌ NO'}")
    print(f"All gaps from binance_vision: {'✅ YES' if not any(df[df['open_time_utc'].isin(missing_index)].shape[0] > 0 for _ in [1]) else '✅ YES (gaps = missing rows, no source)'}")
    print(f"Any gaps in 2024+: {'❌ YES — investigate!' if any(s.year >= 2024 for s, _ in intervals) else '✅ NO'}")

print("\n" + "=" * 70)
print("PHASE 0 SPOT CHECK COMPLETE")
print("=" * 70)
print("\nIf all events pass and yearly trajectory looks correct,")
print("the dataset is verified for Phase 1 backtesting.")



SPOT CHECK 4: GAP DOCUMENTATION FOR KNOWN ANOMALIES

31 missing hours in 15 gap windows:

#    Start (UTC)                  End (UTC)                    Hours  Likely Cause
------------------------------------------------------------------------------------------
1    2020-02-09 02:00:00+00:00    2020-02-09 02:00:00+00:00    1      Missing data (unknown)
2    2020-02-19 12:00:00+00:00    2020-02-19 16:00:00+00:00    5      Missing data (unknown)
3    2020-03-04 10:00:00+00:00    2020-03-04 10:00:00+00:00    1      Missing data (unknown)
4    2020-04-25 02:00:00+00:00    2020-04-25 03:00:00+00:00    2      Missing data (unknown)
5    2020-06-28 02:00:00+00:00    2020-06-28 04:00:00+00:00    3      Missing data (unknown)
6    2020-11-30 06:00:00+00:00    2020-11-30 06:00:00+00:00    1      Missing data (unknown)
7    2020-12-21 15:00:00+00:00    2020-12-21 17:00:00+00:00    3      Exchange downtime
8    2020-12-25 02:00:00+00:00    2020-12-25 02:00:00+00:00    1      Missing data (unkn